# 01 - Document Loader


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand what a `Document` object contains.
- Load TXT, a directory of files, and PDF files.
- Preserve metadata such as source and page.
- Understand why document loading is the first stage of RAG.

## Core AI concept

A document loader converts raw external content into a consistent representation the retrieval system can process. In RAG, loading is not just 'read the file': it is also metadata capture. Metadata later supports filtering, citations, permissions, and debugging.

## Architecture / mental model

```text
Files / Web / DB
      ↓
Document Loader
      ↓
Document(page_content + metadata)
      ↓
Text Splitter
      ↓
Embeddings / Retrieval
```

## Connection to the supplied learning material

Your supplied `rag.ipynb` reads text files from a directory and attaches `source` metadata. The expanded notebook keeps that idea and adds common LangChain loaders.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

### Install loaders

In [ ]:
%pip install -qU langchain-community pypdf pymupdf python-dotenv

### Create sample files

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

(data_dir / "python.txt").write_text(
    "Python is a programming language used for automation, data science, machine learning, and AI.",
    encoding="utf-8",
)
(data_dir / "rag.txt").write_text(
    "RAG means Retrieval-Augmented Generation. It retrieves relevant context before generation.",
    encoding="utf-8",
)
print(list(data_dir.iterdir()))

### Load a single text file

In [ ]:
from langchain_community.document_loaders import TextLoader

docs = TextLoader("data/python.txt", encoding="utf-8").load()
print(docs[0].page_content)
print(docs[0].metadata)

### Load a directory

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()

for doc in docs:
    print("SOURCE:", doc.metadata.get("source"))
    print("TEXT:", doc.page_content)
    print("-" * 60)

### PDF loader example

In [ ]:
# Requires a real PDF file at data/example.pdf
from langchain_community.document_loaders import PyMuPDFLoader

# pdf_docs = PyMuPDFLoader("data/example.pdf").load()
# print(pdf_docs[0].metadata)
# print(pdf_docs[0].page_content[:500])

## Evaluation

Check that every expected source file was loaded and that metadata is present. For PDFs, verify that page metadata survives. A simple loader test is: expected file count == loaded file count.

## Improvement ideas

Use file-type-specific loaders, remove empty documents, normalize encoding, preserve page/section metadata, reject unsupported file types, and record ingestion errors instead of silently dropping files.

## Practice exercises

1. Add DOCX and Markdown loaders.
2. Add a `file_type` metadata field.
3. Build a function that prints a document inventory before indexing.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- LangChain document loaders reference: https://reference.langchain.com/python/langchain-community/document_loaders